In [2]:
%pip install lightgbm

  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
Using cached lightgbm-4.7.0-py3-none-win_amd64.whl (1.4 MB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys

print(sys.executable)

# 11 - Stage 2 LightGBM Comparison

This notebook compares a regularized LightGBM conversation-risk model with the Stage 2 logistic baselines from notebook 10. Only stage and behavioral numeric features are used, so source vocabulary cannot dominate this comparison.

If LightGBM is unavailable, the notebook stops with an explicit package-install instruction rather than silently substituting a different model.

In [3]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

try:
    from lightgbm import LGBMClassifier
except ImportError as exc:
    raise ImportError("Install LightGBM with `pip install lightgbm` before running notebook 11.") from exc

RANDOM_SEED = 42
WORKSPACE = Path.cwd()
PROCESSED_DIR = WORKSPACE / "data" / "processed"
REPORT_DIR = WORKSPACE / "reports"
MODEL_DIR = WORKSPACE / "models"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

features_path = PROCESSED_DIR / "pan12_conversation_features.csv"
if not features_path.exists():
    raise FileNotFoundError("Run 09_pan12_stage2_features.ipynb first.")

conversations = pd.read_csv(features_path)
stage_columns = [
    column for column in [
        "fraction_predicted_200", "fraction_predicted_600", "fraction_predicted_900",
        "stage_600_threshold_onset", "stage_900_threshold_onset",
        "mean_stage_600_probability", "mean_stage_900_probability",
        "max_stage_600_probability", "max_stage_900_probability",
    ] if column in conversations.columns
]
behavioral_columns = [
    column for column in conversations.columns
    if column not in {"split", "conversation_id", "conversation_text", "label", *stage_columns}
]
print("Stage features:", len(stage_columns))
print("Behavioral features:", len(behavioral_columns))

Stage features: 9
Behavioral features: 8


## Model and threshold selection

The validation threshold is chosen on an internal training-only split, then frozen for the supplied test split.

In [4]:
train_frame = conversations.loc[conversations["split"] == "train"].copy()
test_frame = conversations.loc[conversations["split"] == "test"].copy()
train_inner, validation_inner = train_test_split(
    train_frame, test_size=0.20, random_state=RANDOM_SEED, stratify=train_frame["label"]
)

def build_lightgbm(columns):
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("classifier", LGBMClassifier(
            objective="binary", boosting_type="gbdt", class_weight="balanced",
            n_estimators=250, learning_rate=0.05, num_leaves=15,
            min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
            reg_lambda=1.0, random_state=RANDOM_SEED, verbosity=-1,
            n_jobs=-1,
        )),
    ])

def choose_threshold(model, validation_frame, columns):
    scores = model.predict_proba(validation_frame[columns])[:, 1]
    precision, recall, thresholds = precision_recall_curve(validation_frame["label"], scores)
    f1_values = np.divide(
        2 * precision[:-1] * recall[:-1],
        precision[:-1] + recall[:-1],
        out=np.zeros_like(precision[:-1]),
        where=(precision[:-1] + recall[:-1]) != 0,
    )
    return float(thresholds[int(np.argmax(f1_values))])

## Ablation results

Stage-only, behavior-only, and combined numeric variants are evaluated consistently with notebook 10.

In [5]:
variants = {
    "stage_only": stage_columns,
    "behavior_only": behavioral_columns,
    "all_numeric": stage_columns + behavioral_columns,
}

results = []
models = {}
for name, columns in variants.items():
    if len(columns) == 0:
        continue
    model = build_lightgbm(columns)
    model.fit(train_inner[columns], train_inner["label"])
    threshold = choose_threshold(model, validation_inner, columns)
    scores = model.predict_proba(test_frame[columns])[:, 1]
    predictions = (scores >= threshold).astype(int)
    labels = test_frame["label"].to_numpy()
    results.append({
        "variant": name,
        "threshold": threshold,
        "pr_auc": average_precision_score(labels, scores),
        "roc_auc": roc_auc_score(labels, scores),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
    })
    models[name] = model

results_frame = pd.DataFrame(results)
display(results_frame)
results_frame.to_csv(REPORT_DIR / "stage2_lightgbm_results.csv", index=False)
joblib.dump(models["all_numeric"], MODEL_DIR / "stage2_lightgbm_all_numeric.joblib")
with (REPORT_DIR / "stage2_lightgbm_thresholds.json").open("w", encoding="utf-8") as handle:
    json.dump({row["variant"]: row["threshold"] for row in results}, handle, indent=2)

,variant,threshold,pr_auc,roc_auc,precision,recall,f1
0,stage_only,0.844819,0.303866,0.864434,0.349525,0.344661,0.347076
1,behavior_only,0.881573,0.459547,0.955919,0.380990,0.510570,0.436364
2,all_numeric,0.912283,0.565975,0.968115,0.583445,0.465882,0.518078


## Interpretation caveat

Tree models can fit sparse PAN-12 datasets aggressively. Compare their PR-AUC and thresholded metrics against notebook 10, inspect feature importance for dataset artifacts, and do not treat a test-table gain as proof of real-world generalization.